# Compute abundance change for each allele

This script takes in the single-cell FACS data as input, applies various filtering criteria, and then produces a z-score for the abundance of each allele relative to the reference on the same plate.

In [1]:
import pandas as pd
import numpy as np
from random import choice
import matplotlib.pyplot as plt
import polars as pl
import process_dualipa as proc

In [2]:
dualipa_inputs = "../1_inputs"
dualipa_outputs = "../3_outputs"
meta_outputs = "../../../1_allele_collection/3_outputs"

n_cell_threshold = 800
wt_gfp_threshold = 100

In [3]:
# Read in single-cell measurements
pDEST_DUAL_df = pd.read_parquet(f"{dualipa_outputs}/facs_single_cell.parquet")

# Filter to only keep wells with more than 800 cells
pDEST_DUAL_df = pDEST_DUAL_df[pDEST_DUAL_df['n_cells'] > n_cell_threshold]

In [4]:
# get the mean and median measurements per well
keep_cols = ['symbol', 'node_type', 'nt_change','aa_change', 'pla', 'well', 'coordinates', 'n_cells', 'orf_id', 'mut_id', 'valid_well']

pDEST_DUAL_avg_df = pDEST_DUAL_df[keep_cols + ['avg_gfp', 'avg_mcherry','avg_GFP_mCherry_ratio']].drop_duplicates()
pDEST_DUAL_median_df = pDEST_DUAL_df[keep_cols + ['median_gfp', 'median_mcherry','median_GFP_mCherry_ratio']].drop_duplicates()

In [5]:
# _d objects: mean or median GFP_mCherry ratio of each unique (wt orf, plate) combination
# _l objects: list of WT:WT abundances (GFP:mCherry ratios) to estimate assay variability
wt_avg_d, wt_ratio_l = proc.get_wt_variability_d(pDEST_DUAL_avg_df, pDEST_DUAL_df)
wt_median_d, wt_ratio_l_median = proc.get_wt_variability_d_median(pDEST_DUAL_median_df, pDEST_DUAL_df)

In [6]:
wt_std, wt_mean = proc.wt_log2fc_variability(pDEST_DUAL_avg_df, pDEST_DUAL_df)

In [7]:
# Must be two WT replicates on same plate to compute the ratio, otherwise we return a NaN
pDEST_DUAL_avg_allele_df = proc.get_pDEST_DUAL_avg_allele_df(pDEST_DUAL_avg_df, wt_avg_d, wt_ratio_l, wt_std, wt_mean)
pDEST_DUAL_median_allele_df = proc.get_pDEST_DUAL_median_allele_df(pDEST_DUAL_median_df, wt_median_d, wt_ratio_l_median)

In [8]:
# append metadata and write out files
metadata = pl.read_csv(f"{meta_outputs}/slim_metadata.csv").with_columns(
    pl.col("orf_id").cast(pl.Float64).alias("orf_id"),
    pl.col("mut_id").cast(pl.Float64).alias("mut_id")
)

mean_df = pl.DataFrame(pDEST_DUAL_avg_allele_df)
median_df = pl.DataFrame(pDEST_DUAL_median_allele_df)

mean_df = mean_df.join(metadata, on=["orf_id", "mut_id"], how="left")
median_df = median_df.join(metadata, on=["orf_id", "mut_id"], how="left")

mean_df.write_csv(f'{dualipa_outputs}/DUALIPA_mean_zscore.csv')
median_df.write_csv(f'{dualipa_outputs}/DUALIPA_median_zscore.csv')

In [9]:
metadata

orf_id,mut_id,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
f64,f64,str,str,str,str,str,str,f64,f64,str
12573.0,2275.0,"""DES""","""Arg248His""","""743G>A""","""ENSG00000175084""","""Edgotyping3""","""4_VUS""",0.000006,2.0,"""DES_Arg248His"""
1331.0,2595.0,"""GPAM""","""Glu131Gly""","""392A>G""","""ENSG00000119927""","""RC4""",null,null,null,"""GPAM_Glu131Gly"""
52920.0,25131.0,"""LITAF""","""Pro135Ser""","""403C>T""","""ENSG00000189067""","""RC4""","""4_VUS""",null,null,"""LITAF_Pro135Ser"""
70497.0,3786.0,"""AGXT""","""Asn22Ser""","""65A>G""","""ENSG00000172482""","""Edgotyping3""","""2_Benign""",0.006737,2.0,"""AGXT_Asn22Ser"""
2474.0,3724.0,"""ERCC5""","""Thr971Ala""","""2911A>G""","""ENSG00000134899""","""RC4""",null,0.00009,null,"""ERCC5_Thr971Ala"""
…,…,…,…,…,…,…,…,…,…,…
54055.0,0.0,"""ALDH18A1""",null,null,"""ENSG00000059573""",null,null,null,null,"""ALDH18A1"""
54326.0,0.0,"""EDA""",null,null,"""ENSG00000158813""",null,null,null,null,"""EDA"""
56245.0,0.0,"""MAP2K1""",null,null,"""ENSG00000169032""",null,null,null,null,"""MAP2K1"""


In [10]:
pdest_layout_df = pd.read_csv(f"{dualipa_outputs}/dualipa_experimental_layout_seq_conf.csv")
pdest_layout_df

,orf_id,mut_id,node_type,dest_pla_id,dest_pla,dest_well,symbol,dualip_sequenced,dualip_sequence_confirmation_class,aa_change,nt_change,ccsb_mutation_id
0,56360.0,0.0,wt,23,VUSDUALXP_23,A12,NEXN,NaN,NaN,WT,WT,NaN
1,56360.0,0.0,wt,23,VUSDUALXP_23,B12,NEXN,NaN,NaN,WT,WT,NaN
2,56360.0,0.0,wt,23,VUSDUALXP_23,C12,NEXN,NaN,NaN,WT,WT,NaN
3,56360.0,0.0,wt,23,VUSDUALXP_23,D12,NEXN,NaN,NaN,WT,WT,NaN
4,9905.0,0.0,wt,23,VUSDUALXP_23,F02,PNKP,NaN,NaN,WT,WT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2306,NaN,NaN,pdest_empty,20,VUSDUALXP_20,A08,NaN,NaN,NaN,WT,WT,NaN
2307,NaN,NaN,pdest_empty,21,VUSDUALXP_21,A07,NaN,NaN,NaN,WT,WT,NaN
2308,NaN,NaN,pdest_empty,21,VUSDUALXP_21,A08,NaN,NaN,NaN,WT,WT,NaN
2309,NaN,NaN,pdest_empty,22,VUSDUALXP_22,A07,NaN,NaN,NaN,WT,WT,NaN
